[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-04-triggers-scheduling.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Triggers and Scheduling — Time, Event, and Webhook
**certified-journeys / kestra-certified** · Practice · Kestra for Data Engineers

> **Goal for today:** By the end of this notebook you can build and validate Kestra YAML flows that use Schedule triggers (with cron), Webhook triggers, and Flow triggers — and use trigger metadata like `{{ trigger.date }}` to make pipelines idempotent.


In [ ]:
%pip install -q pyyaml croniter


## Step 1 · How Kestra triggers work

A Kestra flow can be started by several trigger types. Each trigger type lives under the `triggers:` block at the flow level.

| Trigger type | When it fires | Key fields |
|---|---|---|
| `io.kestra.plugin.core.trigger.Schedule` | Cron schedule | `cron`, `timezone`, `backfill` |
| `io.kestra.plugin.core.trigger.Webhook` | HTTP POST request | `key` (secret path segment) |
| `io.kestra.plugin.core.trigger.Flow` | Another flow succeeds/fails | `namespace`, `flowId`, `states` |

Every trigger injects a `trigger` variable into Pebble templates. For Schedule triggers this includes `trigger.date` — the scheduled execution time.

```yaml
triggers:
  - id: daily_schedule
    type: io.kestra.plugin.core.trigger.Schedule
    cron: "0 9 * * 1-5"   # weekdays at 09:00
```


In [ ]:
import yaml

def build_schedule_trigger(trigger_id: str, cron: str, timezone: str = "UTC") -> dict:
    """Return a Kestra Schedule trigger dict."""
    return {
        "id": trigger_id,
        "type": "io.kestra.plugin.core.trigger.Schedule",
        "cron": cron,
        "timezone": timezone,
    }


def build_flow(namespace: str, flow_id: str, tasks: list, triggers: list) -> dict:
    """Assemble a minimal Kestra flow definition."""
    return {
        "id": flow_id,
        "namespace": namespace,
        "tasks": tasks,
        "triggers": triggers,
    }


# Build a weekday-morning schedule
schedule_trigger = build_schedule_trigger(
    trigger_id="weekday_morning",
    cron="0 9 * * 1-5",
    timezone="America/New_York",
)

# Task that uses {{ trigger.date }} for idempotency
etl_task = {
    "id": "run_etl",
    "type": "io.kestra.plugin.scripts.shell.Commands",
    "commands": [
        "echo 'Running ETL for date: {{ trigger.date | date(\"yyyy-MM-dd\") }}'",
        "python etl.py --date {{ trigger.date | date('yyyy-MM-dd') }}",
    ],
}

flow = build_flow(
    namespace="prod.data",
    flow_id="daily_etl",
    tasks=[etl_task],
    triggers=[schedule_trigger],
)

print("=== Schedule Trigger Flow YAML ===")
print(yaml.dump(flow, default_flow_style=False, sort_keys=False))


### What just happened?

- We built a complete Kestra flow YAML programmatically using Python dicts and `yaml.dump`.
- The `cron: "0 9 * * 1-5"` expression fires **Monday–Friday at 09:00** in `America/New_York`.
- **`{{ trigger.date | date("yyyy-MM-dd") }}`** formats the scheduled timestamp as a date string — the same run always processes the same partition, making the flow idempotent.
- The `yaml.dump` output is valid YAML you can paste directly into the Kestra UI.


## Step 2 · Validating and explaining cron expressions

`croniter` parses and iterates cron expressions. Use it to confirm your schedule fires when you expect before pasting into Kestra.

```python
from croniter import croniter
from datetime import datetime

it = croniter("0 9 * * 1-5", datetime.utcnow())
next_run = it.get_next(datetime)
```

Cron field order: **minute · hour · day-of-month · month · day-of-week** (0 = Sunday).


In [ ]:
from croniter import croniter
from datetime import datetime, timezone


def explain_cron(expression: str, n: int = 5, base: datetime | None = None) -> None:
    """Print the next N fire times for a cron expression."""
    base = base or datetime(2024, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
    it = croniter(expression, base)
    print(f"Cron: '{expression}' — next {n} runs (from {base.date()})")
    for _ in range(n):
        print(f"  {it.get_next(datetime).strftime('%Y-%m-%d %H:%M %Z')}")


# Weekday mornings
explain_cron("0 9 * * 1-5")

print()

# Every 15 minutes
explain_cron("*/15 * * * *")

print()

# First day of each month at midnight
explain_cron("0 0 1 * *")


### What just happened?

- `croniter` iterated the schedule and printed the next 5 real timestamps for each expression.
- **Weekday schedule** correctly skips weekends — no Saturday/Sunday dates appear.
- **`*/15`** fires four times per hour, useful for near-realtime ingestion jobs.
- Always run this validation before committing a cron expression to a production flow — a misplaced field order can cause hourly runs instead of daily, or vice versa.


## Step 3 · Backfill execution

Kestra Schedule triggers support **backfill** — replaying missed executions for a historical date range. This is critical for recovering from outages or deploying a new pipeline with historical data.

In the Kestra UI: Flow → Triggers → Schedule → "Backfill executions" → pick start/end date.  
Via the API:
```http
POST /api/v1/triggers/{namespace}/{flowId}/{triggerId}/backfill
Body: {"start": "2024-01-01T00:00:00Z", "end": "2024-01-07T00:00:00Z"}
```

Each backfill execution receives the correct `trigger.date` for its slot — idempotent flows process the right partition automatically.


In [ ]:
from datetime import timedelta


def simulate_backfill(
    cron: str,
    start: datetime,
    end: datetime,
) -> list[dict]:
    """
    Simulate which trigger dates Kestra would create for a backfill.
    Returns a list of dicts mimicking trigger context variables.
    """
    it = croniter(cron, start - timedelta(seconds=1))  # include start
    executions = []
    while True:
        ts = it.get_next(datetime)
        if ts > end:
            break
        executions.append({
            "trigger.date": ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "trigger.date_formatted": ts.strftime("%Y-%m-%d"),
            "partition_path": f"s3://datalake/raw/date={ts.strftime('%Y-%m-%d')}/",
        })
    return executions


# Simulate a 5-day backfill for a daily schedule
start_dt = datetime(2024, 1, 1, tzinfo=timezone.utc)
end_dt = datetime(2024, 1, 5, 23, 59, tzinfo=timezone.utc)

runs = simulate_backfill("0 9 * * *", start_dt, end_dt)
print(f"Backfill would create {len(runs)} executions:\n")
for r in runs:
    print(f"  trigger.date={r['trigger.date']}  →  {r['partition_path']}")


### What just happened?

- We simulated the 5 execution slots Kestra would produce for a 5-day backfill.
- Each slot gets a unique `trigger.date` — an idempotent ETL script can write to `s3://datalake/raw/date=YYYY-MM-DD/` without overwriting other partitions.
- **Key insight:** backfill + idempotency = safe replay. The same flow definition handles both live and historical runs.


## Step 4 · Webhook triggers

A Webhook trigger exposes an HTTP endpoint in Kestra. Any system that can POST JSON can kick off a flow — CI/CD pipelines, GitHub Actions, Slack bots, etc.

```yaml
triggers:
  - id: on_push
    type: io.kestra.plugin.core.trigger.Webhook
    key: my_secret_key   # appended to URL: /api/v1/executions/webhook/.../my_secret_key
```

The POST body is available as `{{ trigger.body }}` and individual fields as `{{ trigger.body.field }}`.

**Testing locally with curl:**
```bash
curl -X POST \
  http://localhost:8080/api/v1/executions/webhook/prod.data/deploy_model/on_push \
  -H 'Content-Type: application/json' \
  -d '{"model_version": "v2.1", "commit": "abc123"}'
```


In [ ]:
import json


def build_webhook_trigger(trigger_id: str, secret_key: str) -> dict:
    """Return a Kestra Webhook trigger dict."""
    return {
        "id": trigger_id,
        "type": "io.kestra.plugin.core.trigger.Webhook",
        "key": secret_key,
    }


def render_webhook_url(namespace: str, flow_id: str, secret_key: str,
                        host: str = "http://localhost:8080") -> str:
    """Return the webhook endpoint URL."""
    return f"{host}/api/v1/executions/webhook/{namespace}/{flow_id}/{secret_key}"


# Build a model-deploy flow triggered by CI/CD
webhook_trigger = build_webhook_trigger(
    trigger_id="on_ci_push",
    secret_key="ci_secret_xyz",
)

deploy_task = {
    "id": "deploy_model",
    "type": "io.kestra.plugin.scripts.shell.Commands",
    "commands": [
        "echo 'Deploying version: {{ trigger.body.model_version }}'",
        "echo 'Commit: {{ trigger.body.commit }}'",
        "./deploy.sh --version {{ trigger.body.model_version }}",
    ],
}

webhook_flow = build_flow(
    namespace="prod.ml",
    flow_id="deploy_model",
    tasks=[deploy_task],
    triggers=[webhook_trigger],
)

print("=== Webhook Trigger Flow YAML ===")
print(yaml.dump(webhook_flow, default_flow_style=False, sort_keys=False))

print("=== Endpoint URL ===")
print(render_webhook_url("prod.ml", "deploy_model", "ci_secret_xyz"))

print("\n=== curl test command ===")
payload = {"model_version": "v2.1", "commit": "abc123"}
url = render_webhook_url("prod.ml", "deploy_model", "ci_secret_xyz")
print(f"curl -X POST {url} \\")
print(f"  -H 'Content-Type: application/json' \\")
print(f"  -d '{json.dumps(payload)}'")


### What just happened?

- The Webhook trigger uses a `key` field that forms the last segment of the endpoint URL — keep it secret (use a Kestra secret or env var in production).
- **`{{ trigger.body.model_version }}`** reads directly from the POST JSON body — no extra input declarations needed.
- The `curl` command shows exactly how a CI/CD system would kick off this flow after building and pushing a model artifact.
- In production, store the webhook key in Kestra Secrets: `{{ secret('WEBHOOK_KEY') }}`.


## Step 5 · Flow triggers — event-driven pipelines

A Flow trigger lets a **downstream** flow react to the terminal state of an **upstream** flow. This replaces manual chaining or polling.

```yaml
triggers:
  - id: on_upstream_success
    type: io.kestra.plugin.core.trigger.Flow
    namespace: prod.data
    flowId: daily_etl
    states:
      - SUCCESS
```

Available states to listen on: `SUCCESS`, `FAILED`, `WARNING`, `KILLED`.

The downstream flow has access to `{{ trigger.executionId }}` and `{{ trigger.namespace }}`.


In [ ]:
def build_flow_trigger(
    trigger_id: str,
    upstream_namespace: str,
    upstream_flow_id: str,
    states: list[str] | None = None,
) -> dict:
    """Return a Kestra Flow trigger dict."""
    trigger = {
        "id": trigger_id,
        "type": "io.kestra.plugin.core.trigger.Flow",
        "namespace": upstream_namespace,
        "flowId": upstream_flow_id,
    }
    if states:
        trigger["states"] = states
    return trigger


# Downstream reporting flow — fires when daily_etl succeeds
flow_trigger = build_flow_trigger(
    trigger_id="on_etl_success",
    upstream_namespace="prod.data",
    upstream_flow_id="daily_etl",
    states=["SUCCESS"],
)

report_task = {
    "id": "generate_report",
    "type": "io.kestra.plugin.scripts.python.Script",
    "script": (
        "print('Upstream execution: {{ trigger.executionId }}')\n"
        "print('Generating daily report...')\n"
    ),
}

downstream_flow = build_flow(
    namespace="prod.reporting",
    flow_id="daily_report",
    tasks=[report_task],
    triggers=[flow_trigger],
)

print("=== Flow Trigger (downstream) YAML ===")
print(yaml.dump(downstream_flow, default_flow_style=False, sort_keys=False))

# Show the relationship between flows
print("=== Pipeline topology ===")
print(f"  [prod.data / daily_etl]  ──SUCCESS──>  [prod.reporting / daily_report]")
print(f"  Trigger type : io.kestra.plugin.core.trigger.Flow")
print(f"  Listens for  : {flow_trigger['states']}")


### What just happened?

- The downstream flow declares a `Flow` trigger — Kestra subscribes it to the execution events of `daily_etl`.
- When `daily_etl` reaches the `SUCCESS` state, Kestra automatically creates a new execution of `daily_report`.
- **No polling, no cron offset hacks** — the downstream flow starts the instant the upstream one finishes.
- `{{ trigger.executionId }}` is useful for audit trails and linking downstream reports back to the source run.


## Step 6 · Combining all three trigger types in one flow

A single Kestra flow can have **multiple triggers** of different types. This is useful for flows that should run on a schedule but can also be kicked off manually via webhook or by another flow.

```yaml
triggers:
  - id: scheduled
    type: io.kestra.plugin.core.trigger.Schedule
    cron: "0 6 * * *"
  - id: on_demand
    type: io.kestra.plugin.core.trigger.Webhook
    key: manual_run_key
  - id: on_upstream
    type: io.kestra.plugin.core.trigger.Flow
    namespace: prod.ingest
    flowId: load_raw_data
    states: [SUCCESS]
```


In [ ]:
def validate_triggers(flow: dict) -> list[str]:
    """
    Validate that every trigger in the flow has required fields.
    Returns a list of validation errors (empty list = valid).
    """
    errors = []
    required_by_type = {
        "io.kestra.plugin.core.trigger.Schedule": ["cron"],
        "io.kestra.plugin.core.trigger.Webhook": ["key"],
        "io.kestra.plugin.core.trigger.Flow": ["namespace", "flowId"],
    }
    for t in flow.get("triggers", []):
        tid = t.get("id", "<no-id>")
        ttype = t.get("type", "")
        if ttype not in required_by_type:
            errors.append(f"[{tid}] Unknown trigger type: {ttype}")
            continue
        for field in required_by_type[ttype]:
            if field not in t:
                errors.append(f"[{tid}] Missing required field: '{field}'")
    return errors


# Build a flow with all three trigger types
multi_trigger_flow = build_flow(
    namespace="prod.data",
    flow_id="transform_and_report",
    tasks=[
        {
            "id": "transform",
            "type": "io.kestra.plugin.scripts.shell.Commands",
            "commands": ["python transform.py"],
        }
    ],
    triggers=[
        build_schedule_trigger("nightly", "0 2 * * *"),
        build_webhook_trigger("on_demand", "manual_abc123"),
        build_flow_trigger("on_ingest", "prod.ingest", "load_raw_data", ["SUCCESS"]),
    ],
)

print("=== Multi-Trigger Flow YAML ===")
print(yaml.dump(multi_trigger_flow, default_flow_style=False, sort_keys=False))

print("=== Validation ===")
errors = validate_triggers(multi_trigger_flow)
if errors:
    for e in errors:
        print(f"  ERROR: {e}")
else:
    print(f"  All {len(multi_trigger_flow['triggers'])} triggers are valid.")

# Assert structure is correct
assert len(multi_trigger_flow["triggers"]) == 3
trigger_types = [t["type"] for t in multi_trigger_flow["triggers"]]
assert "io.kestra.plugin.core.trigger.Schedule" in trigger_types
assert "io.kestra.plugin.core.trigger.Webhook" in trigger_types
assert "io.kestra.plugin.core.trigger.Flow" in trigger_types
print("\n  Assertions passed — all trigger types present.")


### What just happened?

- We built a flow with three distinct trigger types and validated each one against its required fields.
- The `validate_triggers` function mirrors the checks Kestra performs at flow registration time.
- **Python assertions** confirm the generated YAML structure is correct — a lightweight test harness for flow generation code.
- In production, generating flows programmatically (e.g. from a YAML template or a dbt manifest) and validating before pushing to Kestra is a solid CI/CD pattern.


## Step 7 · Using `trigger.date` for idempotent partitioning

Kestra's Pebble templating provides date formatting filters. The most important for ETL:

| Expression | Output example |
|---|---|
| `{{ trigger.date \| date('yyyy-MM-dd') }}` | `2024-01-15` |
| `{{ trigger.date \| date('yyyy/MM') }}` | `2024/01` |
| `{{ trigger.date \| date('HH') }}` | `09` (hour) |
| `{{ trigger.date \| dateAdd(-1, 'DAYS') \| date('yyyy-MM-dd') }}` | `2024-01-14` (yesterday) |

Use these to build partition paths, file names, and SQL WHERE clauses that are tied to the scheduled slot — not to wall-clock `now()`.


In [ ]:
def render_pebble_date(trigger_date: datetime, template: str) -> str:
    """
    Simulate Kestra's Pebble date filter using Python strftime.
    Maps common Pebble format patterns to Python strftime codes.
    """
    # Map Pebble Java date format → Python strftime
    fmt_map = {
        "yyyy-MM-dd": "%Y-%m-%d",
        "yyyy/MM": "%Y/%m",
        "HH": "%H",
        "yyyyMMdd": "%Y%m%d",
        "yyyy-MM-dd'T'HH:mm:ss": "%Y-%m-%dT%H:%M:%S",
    }
    py_fmt = fmt_map.get(template, "%Y-%m-%d")
    return trigger_date.strftime(py_fmt)


def generate_idempotent_commands(trigger_date: datetime) -> list[str]:
    """Generate shell commands that would be rendered in a Kestra task."""
    date_str = render_pebble_date(trigger_date, "yyyy-MM-dd")
    month_str = render_pebble_date(trigger_date, "yyyy/MM")

    return [
        f"python extract.py --date {date_str}",
        f"aws s3 sync /tmp/data/ s3://lake/raw/{month_str}/date={date_str}/",
        f"dbt run --vars '{{\"run_date\": \"{date_str}\"}}' --select tag:daily",
    ]


# Simulate trigger dates for a backfill
base_date = datetime(2024, 3, 18, 9, 0, 0, tzinfo=timezone.utc)
print("Rendered task commands per trigger slot:\n")
for offset in range(3):
    slot = base_date - timedelta(days=offset)
    print(f"  Slot: {slot.strftime('%Y-%m-%d')}")
    for cmd in generate_idempotent_commands(slot):
        print(f"    $ {cmd}")
    print()

# Validate that each slot produces a unique date string
slots = [base_date - timedelta(days=i) for i in range(5)]
date_strings = [render_pebble_date(s, "yyyy-MM-dd") for s in slots]
assert len(set(date_strings)) == 5, "All date strings must be unique"
print("Assertion passed: all 5 slots produce unique date strings.")


### What just happened?

- We simulated how Kestra's Pebble `| date(...)` filter resolves to a concrete string for each scheduled slot.
- **Every command targets a specific date partition** — re-running any slot overwrites only that partition, not others.
- The assertion confirms no two slots share the same date string, which is the precondition for idempotent partitioning.
- **Tip:** When using `trigger.date` in dbt vars or SQL, the pipeline becomes a pure function of time — safe to replay, test in isolation, and audit.


In [ ]:
# ─── Challenge ────────────────────────────────────────────────────────────────
# Challenge: Generate a Kestra flow YAML that has:
#   1. A Schedule trigger firing every Monday at 06:00 UTC
#   2. A Webhook trigger with key "weekly_manual"
#   3. A task that prints the date partition path:
#      s3://reports/weekly/year={{ trigger.date | date('yyyy') }}/week=<ISO week number>
#
# Bonus: use croniter to print the next 4 Monday run times.
# Validate the YAML structure with validate_triggers.
# ──────────────────────────────────────────────────────────────────────────────

# TODO: define the Schedule trigger (cron for every Monday at 06:00)
weekly_schedule = None

# TODO: define the Webhook trigger
weekly_webhook = None

# TODO: define the task (use io.kestra.plugin.scripts.shell.Commands)
weekly_task = None

# TODO: assemble with build_flow and print yaml.dump
# weekly_flow = build_flow(...)

# TODO: validate and print errors (should be empty)
# errors = validate_triggers(weekly_flow)

# TODO: use explain_cron to print next 4 Monday run times
# explain_cron("...", n=4)


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| Schedule trigger | Fires on a cron expression; supports `timezone` and `backfill` |
| cron format | `minute hour day month weekday` — validate with `croniter` before deploying |
| Backfill | Replays missed slots; each gets the correct `trigger.date` automatically |
| Webhook trigger | HTTP POST to `/api/v1/executions/webhook/.../KEY`; body in `{{ trigger.body }}` |
| Flow trigger | Fires when upstream flow reaches a specified state (`SUCCESS`, `FAILED`, etc.) |
| `{{ trigger.date }}` | Scheduled timestamp; use `\| date('yyyy-MM-dd')` for idempotent partitioning |
| Multiple triggers | A single flow can have Schedule + Webhook + Flow triggers simultaneously |

> **Tip:** Use `{{ trigger.date | date('yyyy-MM-dd') }}` to make tasks idempotent — the same output regardless of when triggered.

---
## What's next
**Day 5** → Inputs, Outputs, and Passing Data Between Tasks — define typed inputs, chain task outputs through Pebble templates, and use internal storage for large files.

Mark Day 4 complete in your [tracker](../index.html).
